In [11]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=0c586cb85d676f88e0131037df159a0d1ddf9d7ecf1595d8f5c2c06eb87dce20
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [12]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



## BB84 Protocol Simulation (Without Attacker)

### Alice's Preparations

In [13]:
from qiskit import QuantumCircuit
from qiskit.providers.basic_provider import BasicSimulator

# Helper function to generate a random bit using quantum measurement
def get_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)  # Apply Hadamard to create superposition
    qc.measure(0, 0) # Measure in computational basis
    simulator = BasicSimulator()
    compiled_circuit = transpile(qc, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc)
    return int(list(counts.keys())[0]) # Returns 0 or 1

In [14]:
KEY_LENGTH = 20 # Number of qubits Alice will send

# Alice generates her random bits (key) and random basis choices
alice_bits = [get_random_bit() for _ in range(KEY_LENGTH)]
alice_bases = [get_random_bit() for _ in range(KEY_LENGTH)] # 0 for Z (computational), 1 for X (Hadamard)

print(f"Alice's generated bits: {alice_bits}")
print(f"Alice's chosen bases (0=Z, 1=X): {alice_bases}")

Alice's generated bits: [1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0]
Alice's chosen bases (0=Z, 1=X): [0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0]


### Alice Encodes Qubits

In [15]:
from qiskit import QuantumCircuit

alice_qubits = []
for i in range(KEY_LENGTH):
    qc = QuantumCircuit(1, 1) # One qubit, one classical bit
    bit = alice_bits[i]
    basis = alice_bases[i]

    if bit == 1:
        qc.x(0) # If bit is 1, flip it from |0> to |1>

    if basis == 1:
        qc.h(0) # If basis is X, apply Hadamard

    alice_qubits.append(qc)

print(f"Alice has prepared {len(alice_qubits)} qubits for sending.")

Alice has prepared 20 qubits for sending.


### Bob's Actions

In [16]:
# Bob generates his random basis choices
bob_bases = [get_random_bit() for _ in range(KEY_LENGTH)] # 0 for Z (computational), 1 for X (Hadamard)

print(f"Bob's chosen bases (0=Z, 1=X): {bob_bases}")

Bob's chosen bases (0=Z, 1=X): [0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1]


In [17]:
from qiskit.providers.basic_provider import BasicSimulator

bob_measurements = []
simulator = BasicSimulator()

for i in range(KEY_LENGTH):
    qc_bob = alice_qubits[i].copy() # Bob receives Alice's qubit circuit
    basis = bob_bases[i]

    if basis == 1:
        qc_bob.h(0) # If Bob chooses X basis, apply Hadamard before measuring

    qc_bob.measure(0, 0) # Measure the qubit

    compiled_circuit = transpile(qc_bob, simulator)
    job = simulator.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts(qc_bob)
    bob_measurements.append(int(list(counts.keys())[0]))

print(f"Bob's measurements: {bob_measurements}")

Bob's measurements: [1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0]


### Sifting and Key Generation

In [18]:
sifted_alice_key = []
sifted_bob_key = []

for i in range(KEY_LENGTH):
    if alice_bases[i] == bob_bases[i]:
        # Bases match, keep the bit
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_measurements[i])

print(f"Sifted Alice's key: {sifted_alice_key}")
print(f"Sifted Bob's key: {sifted_bob_key}")

Sifted Alice's key: [1, 0, 1, 1, 1, 0, 0, 0, 0, 1]
Sifted Bob's key: [1, 0, 1, 1, 1, 0, 0, 0, 0, 1]


### Key Reconciliation and Error Checking

In [19]:
errors = 0
for i in range(len(sifted_alice_key)):
    if sifted_alice_key[i] != sifted_bob_key[i]:
        errors += 1

error_rate = errors / len(sifted_alice_key) if len(sifted_alice_key) > 0 else 0

print(f"Number of matching bits in sifted key: {len(sifted_alice_key)}")
print(f"Number of errors (disrupted qubits): {errors}")
print(f"Error rate: {error_rate:.2f}")

if errors == 0:
    print("\nBB84 Protocol successful! Alice and Bob share an identical key, and no attacker was detected.")
    final_key = sifted_alice_key # Or sifted_bob_key, they are identical
    print(f"Final shared key: {final_key}")
else:
    print("\nErrors detected! There might have been an eavesdropper or channel noise.")
    print("Alice and Bob discard the key and restart the protocol.")

Number of matching bits in sifted key: 10
Number of errors (disrupted qubits): 0
Error rate: 0.00

BB84 Protocol successful! Alice and Bob share an identical key, and no attacker was detected.
Final shared key: [1, 0, 1, 1, 1, 0, 0, 0, 0, 1]
